# 03 — Experimentos preliminares: Isolation Forest e indicadores físicos

Este notebook reproduz os experimentos exploratórios **anteriores** à regra final: janelas de 10 minutos, comparação por regime e atributos de 60 minutos. Esses modelos não comprovaram antecipação persistente de F2–F4 na configuração original. É um histórico importante do desenvolvimento da pesquisa.

Requer o CSV completo e as dependências instaladas.

In [ ]:
# Preparação: se executado a partir de um notebook aberto no Colab,
# carregue o ZIP do repositório gerado neste projeto (ou clone-o via GitHub).
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = ""  # após publicar, opcional: "https://github.com/SEU_USUARIO/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

In [ ]:
# CSV público original: o arquivo NÃO acompanha o GitHub.
# No Colab, recomendamos colocá-lo no Google Drive para evitar reenvios.
from pathlib import Path
CSV = BASE / 'dados' / 'MetroPT3(AirCompressor).csv'
if not CSV.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        candidato = Path('/content/drive/MyDrive/MetroPT3(AirCompressor).csv')
        if candidato.exists(): CSV = candidato
    except ImportError:
        pass
assert CSV.exists(), 'Coloque MetroPT3(AirCompressor).csv em dados/ ou na raiz do Meu Drive.'
print('CSV:', CSV, 'MB:', round(CSV.stat().st_size/1e6, 1))

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','scikit-learn','matplotlib'],check=True)
E1=BASE/'resultados'/'etapa1'; E2=BASE/'resultados'/'etapa2'; E3=BASE/'resultados'/'etapa3'
for d in (E1,E2,E3):d.mkdir(parents=True,exist_ok=True)

## 1. Isolation Forest inicial: fevereiro para treinamento

In [ ]:
subprocess.run([sys.executable,'analises/modelo_exploratorio_etapa1.py','--csv',str(CSV),'--out',str(E1)],check=True)

## 2. Comparação de modelos globais e por regime

In [ ]:
subprocess.run([sys.executable,'analises/etapa2_modelos_regime.py','--windows',str(E1/'janelas_10min_baseline_exploratorio.csv'),'--out',str(E2)],check=True)
import pandas as pd
from IPython.display import display
display(pd.read_csv(E2/'comparacao_modelos.csv'))

## 3. Engenharia de indicadores físicos e temporais

In [ ]:
subprocess.run([sys.executable,'analises/etapa3_indicadores_fisicos.py','--csv',str(CSV),'--out',str(E3)],check=True)
display(pd.read_csv(E3/'comparacao_modelos.csv'))